In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
# os.environ['GEMINI_API_KEY']=os.getenv("GEMINI_API_KEY")
os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")
# print("GEMINI_API_KEY:", os.environ['GEMINI_API_KEY'])

from langchain.chat_models import init_chat_model
model = init_chat_model(model="llama-3.1-8b-instant", model_provider="groq",
                            api_key=os.getenv("GROQ_API_KEY"))

# Tool calling

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code.

In [2]:
from langchain.tools import tool

@tool
def fetch_weather_data(location: str) -> str:
    """Fetch current weather data for a given location."""
    return f"The current weather in {location} is sunny with a temperature of 25°C."

# IMPORTANT: bind_tools returns a new runnable; you must assign it
# Try to call the model with tools; if the remote function-call fails (400),
# fall back to running the local tool directly so the notebook continues
# and later cells that inspect response.tool_calls won't error out.
from types import SimpleNamespace

model_with_tools = model.bind_tools([fetch_weather_data], tool_choice='any')

try:
    response = model_with_tools.invoke("What's the weather in New York City?")
except Exception as e:
    # If the model attempted to call a remote function but the platform
    # rejected it (BadRequestError/tool_use_failed), fall back to the local tool.
    print("Tool calling failed; falling back to local tool. Error:", e)
    result = fetch_weather_data("New York City")
    # Provide a minimal object that other cells expect (content and tool_calls).
    response = SimpleNamespace(content=result, tool_calls=[], additional_kwargs={}, response_metadata={})

# Tool Calling in LangChain: Complete Guide

## What are Tools?

Tools are functions that an AI model can call to perform tasks beyond generating text. They bridge the gap between the model's knowledge and external systems like databases, APIs, calculators, or web services.

## How Tool Calling Works

### 1. **Define Tools**
Tools are Python functions decorated with `@tool` from `langchain.tools`. The docstring becomes the tool's description that the model reads:
```python
@tool
def get_weather(location: str) -> str:
    """Get current weather for a location."""
    return f"Weather in {location}: Sunny, 25°C"
```

### 2. **Bind Tools to Model**
Use `.bind_tools()` to give the model access to tools. **IMPORTANT:** This returns a new object that must be assigned:
```python
model_with_tools = model.bind_tools([get_weather, get_time], tool_choice='any')
```

### 3. **Model Processes Query**
When you invoke the model with a query, it decides whether to call a tool:
```python
response = model_with_tools.invoke("What's the weather in London?")
```

### 4. **Check Tool Calls**
Access the tool calls made by the model:
```python
response.tool_calls  # Returns list of ToolCall objects with name, args, id
```

### 5. **Execute Tools**
For each tool call, invoke the tool and collect results:
```python
for tool in response.tool_calls:
    result = get_weather.invoke(tool)  # Executes the tool
    messages.append(result)  # Add result to conversation
```

### 6. **Get Final Response**
Send tool results back to the model for a final, informed response:
```python
final_response = model.invoke(messages)
```

## Tool Design Best Practices

| Practice | Good ✅ | Bad ❌ |
|----------|---------|--------|
| **Specificity** | `multiply_numbers(a, b)` | `calculate(expression)` |
| **Docstring** | "Multiply two numbers. Use ONLY when..." | "Do math" |
| **Parameters** | `number_a: int, number_b: int` | `expression: str` |
| **Purpose** | One clear job | Generic catch-all with `eval()` |

## Common Issues & Solutions

| Issue | Cause | Solution |
|-------|-------|----------|
| Empty `tool_calls` | Result not assigned from `bind_tools()` | Use `model = model.bind_tools(...)` |
| Wrong tool called | Generic docstrings confuse the model | Use specific docstrings with "Use ONLY when..." |
| Tool not found | Whitespace in tool name | Use single-line docstrings |
| API errors (400) | Groq API limitations with tool calling | Use manual tool execution with keyword matching |

## Manual Tool Execution (Workaround)

When the LLM provider doesn't support native tool calling, implement manual execution:

```python
def execute_tool(query: str):
    if 'weather' in query.lower():
        location = extract_location(query)
        return get_weather(location)
    elif 'multiply' in query.lower():
        numbers = extract_numbers(query)
        return multiply_numbers(numbers[0], numbers[1])
```

## Key Takeaways

1. **Tools extend model capabilities** - Provide access to external data and functions
2. **Model chooses tools** - Based on query and tool descriptions
3. **Tool selection matters** - Clear, specific tools with good docstrings lead to correct choices
4. **Conversation loop** - Model → Tool Call → Tool Execution → Model → Final Response
5. **Error handling** - Always handle cases where tools fail or API limitations arise
6. **Assignment is critical** - `bind_tools()` returns new object; must be assigned to variable

# Tool Chaining 

When a model returns tool calls, you need to execute the tools and pass the results back to the model. This creates a conversation loop where the model can use tool results to generate its final response. LangChain includes agent abstractions that handle this orchestration for you.